## 1. Locate the Repository and Import Modules

In [30]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
repo_root = next(
    (
        path
        for path in candidates
        if (path / "tests" / "ggcmi" / "data").exists() and (path / "src" / "modfilegen").exists()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(f"Could not locate the ModFileGen repository from cwd={cwd}.")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from modfilegen import GlobalVariables
from modfilegen.Converter.CelsiusConverter.celsiusconverter import fetch_data_from_sqlite, main

print(f"Kernel cwd: {cwd}")
print(f"Repo root: {repo_root}")

Kernel cwd: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/notebooks
Repo root: /mnt/d/Mes Donnees/TCMP/github/ModFileGen


## 2. Configure Paths

In [31]:
data_dir = repo_root / "tests" / "ggcmi" / "data"
output_dir = repo_root / "tests" /  "ggcmi" / "output_celsius"
temp_dir = output_dir / "temp"

master_input_db = data_dir / "MasterInput_validation.db"
ori_master_input_db = data_dir / "MasterInput_validation.db"
models_dict_db = data_dir / "ModelsDictionaryArise.db"
celsius_db = data_dir / "CelsiusV3nov17_dataArise.db"

output_dir.mkdir(parents=True, exist_ok=True)
temp_dir.mkdir(parents=True, exist_ok=True)

n_threads = 1
n_parts = 1
# dt=0 keeps generated Celsius folders for inspection. Set dt=1 to clean intermediate files after successful runs.
dt = 0

print(f"Master Input DB exists:     {master_input_db.exists()} -> {master_input_db}")
print(f"Ori Master Input DB exists: {ori_master_input_db.exists()} -> {ori_master_input_db}")
print(f"Models Dict DB exists:      {models_dict_db.exists()} -> {models_dict_db}")
print(f"Celsius DB exists:          {celsius_db.exists()} -> {celsius_db}")
print(f"Output directory: {output_dir}")
print(f"Temp directory:   {temp_dir}")

Master Input DB exists:     True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/data/MasterInput_validation.db
Ori Master Input DB exists: True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/data/MasterInput_validation.db
Models Dict DB exists:      True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/data/ModelsDictionaryArise.db
Celsius DB exists:          True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/data/CelsiusV3nov17_dataArise.db
Output directory: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/output_celsius
Temp directory:   /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/output_celsius/temp


## 3. Preview Simulations

In [32]:
rows = fetch_data_from_sqlite(str(master_input_db))
print(f"Total simulations: {len(rows)}")

for index, row in enumerate(rows[:10], start=1):
    print(
        f"{index}. {row['idsim']} | "
        f"site={row.get('idPoint')} start={row.get('StartYear')}-{row.get('StartDay')} "
        f"end={row.get('EndYear')}-{row.get('EndDay')}"
    )

Total simulations: 60
1. ZIMU_1981_1_N0_C0_baseline.2_saws | site=ZIMU start=1981.0-305 end=1982-318
2. ZIMU_1982_1_N0_C0_baseline.2_saws | site=ZIMU start=1982.0-319 end=1983-318
3. ZIMU_1983_1_N0_C0_baseline.2_saws | site=ZIMU start=1983.0-319 end=1984-318
4. ZIMU_1984_1_N0_C0_baseline.2_saws | site=ZIMU start=1984.0-319 end=1985-318
5. ZIMU_1985_1_N0_C0_baseline.2_saws | site=ZIMU start=1985.0-319 end=1986-318
6. ZIMU_1986_1_N0_C0_baseline.2_saws | site=ZIMU start=1986.0-319 end=1987-318
7. ZIMU_1987_1_N0_C0_baseline.2_saws | site=ZIMU start=1987.0-319 end=1988-318
8. ZIMU_1988_1_N0_C0_baseline.2_saws | site=ZIMU start=1988.0-319 end=1989-318
9. ZIMU_1989_1_N0_C0_baseline.2_saws | site=ZIMU start=1989.0-319 end=1990-318
10. ZIMU_1990_1_N0_C0_baseline.2_saws | site=ZIMU start=1990.0-319 end=1991-318


## 4. Set Global Variables

In [33]:
GlobalVariables["dbMasterInput"] = str(master_input_db)
GlobalVariables["dbModelsDictionary"] = str(models_dict_db)
GlobalVariables["dbCelsius"] = str(celsius_db)
GlobalVariables["directorypath"] = str(output_dir)
GlobalVariables["nthreads"] = n_threads
GlobalVariables["dt"] = dt
GlobalVariables["parts"] = n_parts
GlobalVariables["ori_MI"] = str(ori_master_input_db)
GlobalVariables["tempDir"] = str(temp_dir)
GlobalVariables["dailyoutput"] = 1

print("GlobalVariables configured:")
for key, value in GlobalVariables.items():
    print(f"  {key}: {value}")

GlobalVariables configured:
  storeNumMinSimu: 0
  storeNumMaxSimu: 0
  storeKeyDataN: 0
  dbMasterInput: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/data/MasterInput_validation.db
  dbModelsDictionary: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/data/ModelsDictionaryArise.db
  dbCelsius: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/data/CelsiusV3nov17_dataArise.db
  dt: 0
  ori_MI: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/data/MasterInput_validation.db
  parts: 1
  tempDir: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/output_celsius/temp
  package: 
  thirdyear: 0
  dailyoutput: 1
  directorypath: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/output_celsius
  nthreads: 1


## 5. Run the Celsius Converter

In [34]:
print("Starting Celsius conversion...")
print("=" * 60)

try:
    main()
    print("\n" + "=" * 60)
    print("Celsius conversion completed successfully.")
except Exception as error:
    print("\n" + "=" * 60)
    print(f"Error during Celsius conversion: {error}")
    import traceback

    traceback.print_exc()
    raise

Starting Celsius conversion...
📊 Total simulations to process: 60
Processing 1 chunks...
Number of idsims 2220
creating new directory for process
Copy SimulationList
SimulationList copied
Start transfert of climate data from MI to Cel
Start transfert of climate data from MI to Cel
Number of idPoints 1
transfert of climate data from MI to Cel done
copy CropManagement, Soil and SoilLayers
convert celsius
STDERR:
 
STDOUT:
 Start : 2026/06/17 14:43:00
dbMasterInput : URI=file:db/MasterInput.db
dbModelsDictionary : URI=file:db/ModelsDictionaryArise.db
dbCelsius : URI=file:db/CelsiusV3nov17_dataArise.db
n : 10
cmd : convert
models : celsius
nthreads : 1
dbMasterInput : URI=file:/mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/output_celsius/proc_2209afd9-f5a2-4c2c-9572-339dc6f31f9f/MasterInput.db
dbModelsDictionary : URI=file:/mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/data/ModelsDictionaryArise.db
dbCelsius : URI=file:/mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/ggcmi/out

## 6. Inspect Generated Results

In [ ]:
import sqlite3
import pandas as pd

with sqlite3.connect(str(celsius_db)) as conn:
    df_output = pd.read_sql_query("SELECT * FROM OutputSynt", conn)

print(f"Rows in OutputSynt: {len(df_output)}")
display(df_output.head())